In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

fps = []
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if not 'Benign' in filename and not 'Infiltration' in filename:
            fp = os.path.join(dirname, filename)
            print(fp)
            fps.append(fp)
evaluated_attack_classes = [i.split('/')[-1].split('-')[0].lower() for i in fps]

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# CIC-IDS2017: Too easy to reach perfect classification
In this notebook, I use the XGBoost model to classify CIC-IDS2017.
The classification problem is divided into 6 binary classifiaction tasks, one for each of the available attack classes of the dataset. This is done to avoid over-estimating performance on the classes with few 'positive' (malicious) samples.

## Stages
1. an unoptimized XGBoost with sane hyperparam choices
2. hyperparameter search per attack class
3. re-train XGBoost with optimized hyperparameters and use early stopping with an unseen validation set

## Other methodological choices
Mostly data splitting. 25% training, 10% validation, 65% testing. 

This is an uncommon split, typically the training set is the biggest, but for CIC-IDS2017 you really do not need that much data to learn the patterns. 25% is actually already too much, see notebook CIC-IDS2017-3 for an almost equivalent model with minimal training data.

The validation set is hidden until stage 3. The hyperparameter search happens cross-validated on the 25% training data. Every stage is evaluated on the 65% testing data. For even more accurate estimates, the entire procedure could be repeated which would yield different splits.

## Conclusion
It becomes really difficult to estimate the true performance of new, proposed models. Many of them are computationally much more expensive compared to what I'm using here and the remaining margins at the top are tiny.



In [ ]:
import warnings
warnings.simplefilter(action='once', category=FutureWarning)

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, accuracy_score
from statistics import harmonic_mean
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, KFold, train_test_split
from scipy.stats import uniform, randint
from xgboost import XGBClassifier

In [ ]:
dfs = [pd.read_parquet(f) for f in fps] # list > dict of dfs, indexes into way faster, costs readability

In [ ]:
dep = 'Label' # Other than label, no categoricals exist in the cleaned CIC-NIDS features

In [ ]:
conts = dfs[0].columns.difference([dep])

In [ ]:
def xs_y(df_, targ):    
    if not isinstance(targ, list):
        xs = df_[df_.columns.difference([targ])].copy()
    else:
        xs = df_[df_.columns.difference(targ)].copy()
    y = df_[targ].copy()
    return xs, y

In [ ]:
def binarize_label(df, compare_value='Benign'):
    df['Label'] = df['Label'].astype('object')
    # print(df['Label'].value_counts())
    df.loc[df['Label'] != compare_value, 'Label'] = 1
    df.loc[df['Label'] == compare_value, 'Label'] = 0
    # print(df['Label'].value_counts())
    df['Label'] = df['Label'].astype(dtype=np.int32)  
    return df


In [ ]:
for i, df in enumerate(dfs):
    df = binarize_label(df)        
    trn_df, val_df = train_test_split(df, test_size=0.75)
    val_df, test_df = train_test_split(val_df, test_size=0.86667)
    X_train, y_train = xs_y(trn_df, dep)
    X_val, y_val = xs_y(val_df, dep)
    X_test, y_test = xs_y(test_df, dep)
    dfs[i] = (X_train, y_train, X_val, y_val, X_test, y_test)

## XGBoost Stage 1: sane defaults, no hyperparam optimization

In [ ]:
xgb_no_opt_results = []
for i,df_splits in enumerate(dfs):
    print(evaluated_attack_classes[i])
    X_train, y_train, X_val, y_val, X_test, y_test = df_splits
    xgb = XGBClassifier(
        n_estimators= 100,
        use_label_encoder= False,
        max_depth= 8,
        booster= 'gbtree',
        tree_method= 'gpu_hist',
        subsample= 0.5,
        colsample_bytree= 0.5,
        importance_type= 'gain',
        objective='binary:logistic',
        eval_metric='logloss',
        predictor= 'gpu_predictor',
    )

    xgb.fit(X_train, y_train)
    xgb_preds = xgb.predict(X_test)  
        
    xgb_no_opt_results.append([
        evaluated_attack_classes[i],
        'xgb-no-opt',
        X_train.shape,
        X_val.shape,
        X_test.shape,
        round(roc_auc_score(y_true=y_test, y_score=xgb_preds),4),
        round(precision_score(y_true=y_test, y_pred=xgb_preds), 4),
        round(recall_score(y_true=y_test, y_pred=xgb_preds), 4),
        round(f1_score(y_true=y_test, y_pred=xgb_preds), 4)
    ])

In [ ]:
result_df = pd.DataFrame(data=xgb_no_opt_results, columns=['AtkCls', 'model', 'TrainShape', 'ValShape', 'TestShape', 'Auroc', 'Precision', 'Recall', 'F1'])

In [ ]:
result_df

## XGBoost Stage 2: finding hyperparameters

In [ ]:
hyperparams_per_atk_class = []

In [ ]:
xgb_no_opt_results = []
for i,df_splits in enumerate(dfs):    
    print(evaluated_attack_classes[i])
    X_train, y_train, X_val, y_val, X_test, y_test = df_splits
    xgb = XGBClassifier(    
        early_stopping_rounds=None, # Don't stop early, there is friction here between XGB API and SKL API
        # you would need eval_set=[(X_val, y_val)], but that does not work and depending on the implementation may 
        # contribute to an over-optimistic estimate of performance
        use_label_encoder= False,
        colsample_bylevel=1,
        colsample_bynode=1,
        max_bin=256,
        booster= 'gbtree',
        tree_method= 'gpu_hist',
        importance_type='gain',    
        objective='binary:logistic',
        eval_metric='logloss',
        predictor='gpu_predictor',         
        verbosity=0,
        silent=True)
    
    p_grid = {
        "colsample_bytree": uniform(0.7, 0.3),
        "gamma": uniform(0, 0.5),
        "learning_rate": uniform(0.03, 0.3),
        "max_depth": randint(2, 8),
        "n_estimators": randint(100, 150),
        "subsample": uniform(0.6, 0.4)
    }
    
    search = RandomizedSearchCV(xgb, 
                            param_distributions=p_grid,
                            random_state=None, 
                            n_iter=200, 
                            cv=KFold(n_splits=5, shuffle=False),
                            verbose=False)
    
    search.fit(X_train, y_train) 
    hyperparams_per_atk_class.append(search.best_params_)

## XGBoost Stage 3: final classification with optimized parameters.
Now we allow for more boosting rounds and enable early stopping on a validation set which has been kept separate and invisible in the previous stages.

In [ ]:
xgb_opt_results = []
for i,df_splits in enumerate(dfs):    
    print(evaluated_attack_classes[i])
    X_train, y_train, X_val, y_val, X_test, y_test = df_splits
    xgb_earlystop = XGBClassifier(
        n_estimators= 1000,
        early_stopping_rounds=8,
        use_label_encoder= False,
        colsample_bylevel=1,
        colsample_bynode=1,
        max_bin=256,
        booster= 'gbtree',
        tree_method= 'gpu_hist',
        importance_type='gain',    
        objective='binary:logistic',
        eval_metric='logloss',
        predictor='gpu_predictor',
        verbosity=0,
        silent=True
    )
    xgb_earlystop.set_params(**hyperparams_per_atk_class[i])
    # Correct use of a hidden validation set, no leakage, used to stop early
    xgb_earlystop.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)    
    xgb_early_preds = xgb_earlystop.predict(X_test)
    xgb_opt_results.append([
        evaluated_attack_classes[i],
        'xgb-opt',
        X_train.shape,
        X_val.shape,
        X_test.shape,
        round(roc_auc_score(y_true=y_test, y_score=xgb_early_preds),4),
        round(precision_score(y_true=y_test, y_pred=xgb_early_preds), 4),
        round(recall_score(y_true=y_test, y_pred=xgb_early_preds), 4),
        round(f1_score(y_true=y_test, y_pred=xgb_early_preds), 4)
    ])

In [ ]:
optim_result_df = pd.DataFrame(data=xgb_opt_results, columns=['AtkCls', 'model', 'TrainShape', 'ValShape', 'TestShape', 'Auroc', 'Precision', 'Recall', 'F1'])

In [ ]:
optim_result_df